# cBioPortal data extraction tutorial

This notebook demonstrates how to:
1. Connect to the [cBioPortal](https://www.cbioportal.org/) repository for cancer genomics data sets.
2. List and explore cBioPortal studies.
3. View attribute and data type summaries for a selected study.
4. Extract molecular data (mutations, CNA, RNA-seq) and load each into netflow's `Keeper` for further analysis.

## Requires
* Python 3.10
* The [Netflow](https://github.com/areElkin/netflow.git) package

---

## Import required libraries

In [1]:
import pandas as pd

from netflow.prep.extract import CBioPortalClient
from netflow.keepers.keeper import Keeper

## Connect to cBioPortal

In [2]:
cbio = CBioPortalClient()
print('Connected to cBioPortal.')

Connected to cBioPortal.


## View all available studies

In [3]:
all_studies = cbio.list_studies()
print(f'{len(all_studies)} studies available on cBioPortal.\n')

print('First 20 studies:')
for name in list(all_studies)[:20]:
    print(f' * {name}')

535 studies available on cBioPortal.

First 20 studies:
 * Breast Cancer (CPTAC GDC, 2025)
 * Glioma (MSK, Clin Cancer Res 2019)
 * Osteosarcoma (TARGET GDC, 2025)
 * Prostate Cancer MDA PCa PDX (MD Anderson, Clin Cancer Res 2024)
 * Uterine Endometrioid Carcinoma (CPTAC GDC, 2025)
 * Urothelial Carcinoma (Cornell/Trento, Nat Gen 2016)
 * Basal Cell Carcinoma (UNIGE, Nat Genet 2016)
 * Endometrial Carcinoma (TCGA GDC, 2025)
 * Kidney Renal Clear Cell Carcinoma (TCGA, PanCancer Atlas)
 * Adenoid Cystic Carcinoma (JHU, Cancer Prev Res 2016)
 * Uterine Corpus Endometrial Carcinoma (TCGA, PanCancer Atlas)
 * Colorectal Adenocarcinoma (TCGA, Firehose Legacy)
 * Lung Adenocarcinoma (CPTAC GDC, 2025)
 * Breast Invasive Carcinoma (Broad, Nature 2012)
 * Retinoblastoma cfDNA (MSK, Cancer Med 2020)
 * Appendiceal Cancer (MSK, J Clin Oncol 2022)
 * Evolution and co-occurrence of PI3K pathway gene mutations in endometrial carcinoma molecular subtypes at the single-cell level
 * Urothelial Cancer P

In [4]:
# Search for studies by keyword
keyword = 'MSK-IMPACT'
matches = [s for s in all_studies if keyword.lower() in s.lower()]
print(f'Studies matching "{keyword}":')
for m in matches:
    print(f'  {m}')

Studies matching "MSK-IMPACT":
  MSK-IMPACT 50K Clinical Sequencing Cohort (MSK, Cancer Cell 2026)
  MSK-IMPACT Clinical Sequencing Cohort (MSK, Nat Med 2017)
  MSK-IMPACT and MSK-ACCESS Mixed Cohort (MSK, Nat Commun 2021)
  Pan-Cancer MSK-IMPACT MET Validation Cohort (MSK 2022)
  MSK-IMPACT Heme Tumors (MSK, 2022)


## Inspect selected study

In [5]:
STUDY_NAME = 'MSK-IMPACT Clinical Sequencing Cohort (MSK, Nat Med 2017)'

# Study ID and description
study_id = cbio.get_study_id(STUDY_NAME)
print(f'Study ID: {study_id}')

desc = cbio.get_study_description(STUDY_NAME)
print(f'\nDescription:\n{desc[["name", "description", "sequencedSampleCount"]].to_string(index=False)}')

Study ID: msk_impact_2017

Description:
                                                     name                                                             description  sequencedSampleCount
MSK-IMPACT Clinical Sequencing Cohort (MSK, Nat Med 2017) Targeted sequencing of 10,000 clinical cases using the MSK-IMPACT assay                 10945


### > View all molecular data types available in this study

In [6]:
# All molecular data types available in this study
data_types_df = cbio.get_study_attribute(STUDY_NAME, 'data_types')
print('Molecular data profiles available:')
display(data_types_df)

Molecular data profiles available:


,name,id
0,Copy Number Alterations (MSK-IMPACT),msk_impact_2017_cna
1,Mutations (MSK-IMPACT),msk_impact_2017_mutations
2,Structural variants,msk_impact_2017_structural_variants


### > Cancer-type breakdown

In [7]:
cancer_types = cbio.get_study_attribute(STUDY_NAME, 'cancer_types')
print('Top 10 cancer types by sample count:')
display(cancer_types.head(10))

Top 10 cancer types by sample count:


,type,sampleCount
0,Non-Small Cell Lung Cancer,1668
1,Breast Cancer,1337
2,Colorectal Cancer,1007
3,Prostate Cancer,717
4,Glioma,553
5,Pancreatic Cancer,502
6,Soft Tissue Sarcoma,443
7,Bladder Cancer,423
8,Melanoma,365
9,Renal Cell Carcinoma,361


### > Clinical attributes available

In [8]:
clinical_attrs = cbio.list_study_attributes(STUDY_NAME)
print(f'{len(clinical_attrs)} clinical attributes available:')
print(clinical_attrs)

24 clinical attributes available:
['CANCER_TYPE', 'CANCER_TYPE_DETAILED', 'DNA_INPUT', 'FRACTION_GENOME_ALTERED', 'MATCHED_STATUS', 'METASTATIC_SITE', 'MUTATION_COUNT', 'ONCOTREE_CODE', 'OS_MONTHS', 'OS_STATUS', 'PRIMARY_SITE', 'SAMPLE_CLASS', 'SAMPLE_COLLECTION_SOURCE', 'SAMPLE_COUNT', 'SAMPLE_COVERAGE', 'SAMPLE_TYPE', 'SEX', 'SMOKING_HISTORY', 'SOMATIC_STATUS', 'SPECIMEN_PRESERVATION_TYPE', 'SPECIMEN_TYPE', 'TMB_NONSYNONYMOUS', 'TUMOR_PURITY', 'VITAL_STATUS']


## Extract data

### > Clinical data

In [9]:
clin_df = cbio.get_clinical_data(STUDY_NAME)
print(f'Clinical data shape: {clin_df.shape}  (samples × attributes)')
# Display clinical data for first 5 samples
display(clin_df.head(5))

Clinical data shape: (10945, 7)  (samples × attributes)


,patientId,primaryTumorSite,sampleType,tumorPurity,osMonths,osStatus,osGroup
sampleId,,,,,,,
P-0000004-T01-IM3,P-0000004,Breast,Primary,50,NaN,0:LIVING,0.0
P-0000015-T01-IM3,P-0000015,Breast,Metastasis,40,NaN,1:DECEASED,1.0
P-0000023-T01-IM3,P-0000023,Peritoneum,Primary,30,8.71,1:DECEASED,1.0
P-0000024-T01-IM3,P-0000024,Uterus,Metastasis,40,36.75,0:LIVING,0.0
P-0000025-T01-IM3,P-0000025,Uterus,Primary,NaN,8.81,0:LIVING,0.0


### > Extract mutation data and load into Netflow's Keeper

* `get_mutation_data()` returns a binary *(genes, samples)* matrix where 1 indicates gene mutations in a sample , 0 otherwise.   

*Note*: 
* Data pulls may take a few minutes for a large cohort.
* Data extraction utilities like `get_mutation_data()` return the raw data (`mut_raw`) and a processed version (`mut_proc`), structured for simplified import to the Keeper. `mut_proc` is a (n_features, n_observations) (or) (n_genes, n_samples) dataframe.

In [10]:
print('Fetching mutation data...')
mut_raw, mut_proc = cbio.get_mutation_data(STUDY_NAME)

print(f'\nMutation matrix shape: {mut_proc.shape}  (genes × samples)')
print(f'Fraction of gene/sample pairs mutated: {mut_proc.values.mean():.4f}')
display(mut_proc.iloc[:5, :5])

Fetching mutation data...

Mutation matrix shape: (413, 10129)  (genes × samples)
Fraction of gene/sample pairs mutated: 0.0165


,P-0000004-T01-IM3,P-0000015-T01-IM3,P-0000023-T01-IM3,P-0000024-T01-IM3,P-0000025-T01-IM3
hugoGeneSymbol,,,,,
ABL1,0,0,0,0,0
ABRAXAS1,0,0,0,0,0
ACVR1,0,0,0,0,0
AKT1,1,0,0,0,0
AKT2,0,0,0,0,0


In [11]:
# Initialise Keeper with sample IDs from the mutation matrix
sample_ids = mut_proc.columns.tolist()
keeper = Keeper(observation_labels=sample_ids)

# Load mutation data 
keeper.add_data(mut_proc, 'mutation')
print(f'Keeper data keys: {list(keeper.data.keys())}')

Keeper data keys: ['mutation']


In [12]:
# View most frequently mutated genes
mutation_freq = mut_proc.sum(axis=1).sort_values(ascending=False)
print('Top 10 most frequently mutated genes:')
display(mutation_freq.head(10).rename('n_samples_mutated').to_frame())

Top 10 most frequently mutated genes:


,n_samples_mutated
hugoGeneSymbol,
TP53,4538
KRAS,1643
TERT,1460
PIK3CA,1355
APC,1121
ARID1A,875
KMT2D,851
PTEN,665
KMT2C,642


### > Extract CNA data and load it into the Keeper

`get_cna_data()` returns an integer *(genes, samples)*  matrix where -2 represents deep deletion, -1 - shallow deletion, 0 - diploid, 1 - shallow gain, 2-amplification.

In [ ]:
print('Fetching CNA data...')
cna_raw, cna_proc = cbio.get_cna_data(STUDY_NAME)

print(f'\nCNA matrix shape: {cna_raw.shape}  (genes × samples)')
display(cna_proc.head(5))

Fetching CNA data...


In [ ]:
# Align to keeper's sample order before loading
cna_aligned = cna_proc.reindex(columns=sample_ids)
keeper.add_data(cna_aligned, 'cna')
print(f'Keeper data keys: {list(keeper.data.keys())}')

## Keeper summary

In [ ]:
print(f'Keeper observation count: {len(keeper.observation_labels)}')
print(f'Data modalities loaded:   {list(keeper.data.keys())}\n')
for key in keeper.data.keys():
    frame = keeper.data[key].to_frame()
    print(f'  [{key}]  shape: {frame.shape}  dtype: {frame.dtypes.unique()[0]}')